In [6]:
# ==========================================
# PRAKTIKUM BAB 12: NEURAL NETWORK / MLP & CALLBACKS
# irfan dwi fahrezi
# 202343500217
# ==========================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset (menggunakan dataset digits untuk klasifikasi gambar angka sederhana)
data = load_digits()
X, y = data.data, data.target

# Scaling data (sangat penting untuk Neural Network)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("=== PRAKTIKUM 1: EKSPERIMEN ARSITEKTUR MLP ===")
# Praktikum 12.7: Eksperimen berbagai variasi arsitektur hidden layer dan neuron
arsitektur_list = [
    [32],
    [64, 32],
    [128, 64, 32]
]

for arsitektur in arsitektur_list:
    # Membangun model secara dinamis berdasarkan list arsitektur
    layers_list = [keras.layers.Input(shape=(X_train.shape[1],))]
    for n in arsitektur:
        layers_list.append(keras.layers.Dense(n, activation="relu"))
    layers_list.append(keras.layers.Dense(10, activation="softmax")) # 10 kelas output untuk digit

    model = keras.Sequential(layers_list)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    # Latih model
    history = model.fit(X_train, y_train, validation_split=0.2, epochs=30, verbose=0)
    print(f"Arsitektur {str(arsitektur):12} : val_accuracy akhir = {history.history['val_accuracy'][-1]:.3f}")


print("\n=== PRAKTIKUM 2: EARLY STOPPING UNTUK MENCEGAH OVERFITTING ===")
# Praktikum 12.8: Menerapkan Early Stopping callback
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model_es = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

model_es.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history_es = model_es.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    callbacks=[early_stop],
    verbose=0
)

print(f"Pelatihan berhenti pada epoch: {len(history_es.history['loss'])}")
loss_test, acc_test = model_es.evaluate(X_test, y_test, verbose=0)
print(f"Akurasi Test dengan Early Stopping: {acc_test:.3f}")

=== PRAKTIKUM 1: EKSPERIMEN ARSITEKTUR MLP ===
Arsitektur [32]         : val_accuracy akhir = 0.944
Arsitektur [64, 32]     : val_accuracy akhir = 0.944
Arsitektur [128, 64, 32] : val_accuracy akhir = 0.948

=== PRAKTIKUM 2: EARLY STOPPING UNTUK MENCEGAH OVERFITTING ===
Pelatihan berhenti pada epoch: 40
Akurasi Test dengan Early Stopping: 0.978
